# Machine Learning and AI
## Evaluation metrics for final project
Hochschule Campus Wien, Master's program Bioinformatics

Mohamed Goha, BSc. | SS26

This notebook calculates the evaluation metrics for `sample_submission.txt`:

- Frechet ChemNet Distance (FCD) against the precomputed test statistics in `test_stats.p`
- Validity
- Uniqueness
- Novelty

To test your model, create a submission file by putting at least 10000 SMILES strings generated by your model into a file that has the same shape as `sample_submission.txt`. Only the first 10000 will be taken for the evaluation.

`test_stats.p` is effectively your test data. It was pre-calculated using a large number of molecules.


Replace the path in this notebook from `resources/sample_submission.txt` to the path of your submission file to calculate the metrics for your generations.


In [1]:
import pickle
import numpy as np
import pandas as pd

from fcd import canonical_smiles, calculate_frechet_distance, get_predictions, load_ref_model

np.random.seed(1234)

## Load and Canonicalize the Sample Submission

The submission file contains one generated SMILES string per line. Invalid molecules become `None` during canonicalization and are filtered out for the FCD calculation. Please only use rdkit canonicalization.

In [4]:
sample_submission_path = "outputs/submission_t08.txt"

smiles_gen = pd.read_csv(sample_submission_path, header=None)[0].iloc[:10000]
smiles_gen_can = [smi for smi in canonical_smiles(smiles_gen) if smi is not None]

print(f"Generated molecules: {len(smiles_gen)}")
print(f"Valid canonical molecules: {len(smiles_gen_can)}")


Generated molecules: 10000
Valid canonical molecules: 10000


## FCD Against the Test Set

The real test-set statistics were precomputed and saved in `resources/test_stats.p`, so only the generated molecules need to be passed through ChemNet here.

In [6]:
model = load_ref_model()

act_gen = get_predictions(model, smiles_gen_can)
mu_gen = np.mean(act_gen, axis=0)
sigma_gen = np.cov(act_gen.T)

with open("test_stats.p", "rb") as f:
    mu_test, sigma_test = pickle.load(f)

fcd_value = calculate_frechet_distance(
    mu1=mu_gen,
    mu2=mu_test,
    sigma1=sigma_gen,
    sigma2=sigma_test,
)

print("FCD:", fcd_value)


/var/folders/5c/wxv8_vm12sl2dzsgvj_gwlwr0000gp/T/ipykernel_19557/1947466345.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  mu_test, sigma_test = pickle.load(f)


FCD: 1.1485527156614097


## Validity, Uniqueness, and Novelty

## Validity, Uniqueness, and Novelty

In [7]:
smiles_unique = set(smiles_gen_can)

validity = len(smiles_gen_can) / len(smiles_gen)
uniqueness = len(smiles_unique) / len(smiles_gen)

with open("outputs/submission_t08.txt", "r", encoding="utf-8") as f:
    smiles_train = {line.strip() for line in f if line.strip()}

smiles_novel = smiles_unique - smiles_train
novelty = len(smiles_novel) / len(smiles_gen)

print("Validity:", validity)
print("Uniqueness:", uniqueness)
print("Novelty:", novelty)


Validity: 1.0
Uniqueness: 1.0
Novelty: 0.0
